# Notebook 2 - Neural <-> GPT-2 Representational Similarity

**Does the brain organize words like a language model?** (FR1, ds004789; iEEG high-gamma 60-160 Hz.)

We build the super-subject **neural RSM** over 300 words and compare its geometry to GPT-2's
13 layers via **second-order RSA**, with a permutation test for significance. Like Notebook 1,
this starts from the **processed data** - no preprocessing here - and is **pure orchestration**:
every block only calls into `lib/`; no computations or function definitions inline.

**Scope (this draft):** all-regions pooled channels (no per-region breakdown), for both the
**word** window (1.6 s) and the **gap** window (0.5 s). GPT-2 embeddings are **cached to disk**
and reused on re-run.

| Module | Responsibility |
|---|---|
| `lib.config`        | paths (env-var driven), the single subject list |
| `lib.dataio`        | load processed matrices / channels / scores / annot / AAL locations |
| `lib.psth`          | inter-trial reliability score (channel ranking) |
| `lib.supersubject`  | channel selection + super-subject grand matrices |
| `lib.segments`      | word/gap extraction + vocabulary alignment |
| `lib.rsa`           | neural RSM + second-order RSA vs GPT-2 layers |
| `lib.gpt2`          | GPT-2 embeddings (13 layers) + per-layer RSMs |
| `lib.permutation`   | label-shuffle null distribution |
| `lib.viz`           | all plotting helpers |

### Pipeline
```
 1  setup + load GPT-2
 2  load processed data
 3  super-subject + channel selection (180/subj, both sessions)
 4  word/gap extraction + vocabulary alignment   ->  300 common words, identical order
 5  neural RSMs (300x300, words & gaps)           ->  the brain's word-geometry fingerprint
 6  GPT-2 embeddings (13 layers, cached)
 7  GPT-2 per-layer RSMs (300x300 x 13)
 8  second-order RSA: neural vs each layer        ->  Pearson + Spearman, layer profile
 9  permutation test (N=1000)                     ->  empirical p + z per layer
10  outputs
```


## Step 1 - Setup + load GPT-2

**Why:** centralize paths/subjects (portable via `IEEG_DATA_ROOT`) and load the pretrained
GPT-2 once. The model is loaded here so Steps 6-7 can reuse it without re-instantiating.

**Input:** config / environment; the `gpt2` weights (HuggingFace, cached under `~/hf_cache`).

**Output:** `PATHS`, `SUBJECTS`, `RESULTS`/`CACHE` dirs, and `model`/`tokenizer`/`device`.

In [ ]:
import sys
sys.path.append('..')            # repo root on path so `lib` is importable

from lib import config, dataio, psth, supersubject, segments, rsa, gpt2, permutation, viz

PATHS    = config.resolve_paths()
SUBJECTS = config.SUBJECTS
RESULTS  = PATHS.results_dir / 'gpt2'
CACHE    = RESULTS / 'cache'                      # GPT-2 embeddings / null distributions cached here

model, tokenizer, device = gpt2.load_model('gpt2')   # downloads/caches weights on first run
print(f'Subjects : {len(set(s for s, _ in SUBJECTS))} subjects, {len(SUBJECTS)} sessions  |  device: {device}')

## Step 2 - Load processed data

**Why:** load the entry-point artifacts once; this is the boundary that lets the notebook run
without preprocessing.

**Input (per session, from `dr-processed/`):** `_aligned_matrix_NormRatio.npy`,
`_aligned_channels_RAW.pkl`, `correlations_scores_NormRatio.csv`, `.annot`, AAL locations.

**Output:** `DATA` (dict by `{sub}_{ses}`) and per-channel `scores` for ranking.

In [ ]:
DATA   = dataio.load_sessions(SUBJECTS, PATHS)
scores = psth.inter_trial_scores(DATA, recompute=False)
print(f'Loaded {len(DATA)} sessions.')

## Step 3 - Super-subject + channel selection

**Why:** pool all subjects into one MNI super-subject and keep the **180 most reliable channels
per subject** that are present in **both sessions**, z-scored. This is the broad (~4,982-channel)
neural sample the RSM is built on, with identical feature columns across the two days.

**Input:** `DATA`, `scores`; `region='all'`, `top_n=180`.

**Output:** `SEL` (channel-selected, z-scored per-session matrices) + `df_features`.

In [ ]:
SEL, df_features = supersubject.select_channels(
    DATA, scores,
    region    = 'all',     # all-regions pooled (no per-region breakdown in this draft)
    top_n     = 180,
    min_score = -1,
    zscore    = True,
)
print(f'Selected sessions: {len(SEL)} | total channels: {df_features.shape[0]}')

## Step 4 - Word/gap extraction + vocabulary alignment

**Why:** the RSM compares *word pairs*, so every session must contribute the **exact same words
in the same order**. We extract the word (1.6 s) and gap (0.5 s) windows, time-average each to a
scalar per channel, then intersect the vocabularies across all 96 sessions down to the **300
common words** and order them identically before concatenating into the super-subject matrices.

**Input:** `SEL` + `.annot`; word/gap windows.

**Output:** grand matrices `grand.words_s0/s1` and `grand.gaps_s0/s1` (300 words x N channels),
plus `grand.word_labels` (the 300 aligned words).

In [ ]:
segs  = segments.extract_words_and_gaps(
    SEL, PATHS,
    word_window = (0.0, 1.6),    # full word window for the GPT-2 comparison
    gap_window  = (0.0, 0.5),
)

grand = supersubject.build_grand_matrices(segs, df_features, align_vocab=True)   # 300 common words, identical order
print(f'Grand words S0: {grand.words_s0.shape}  |  common words: {len(grand.word_labels)}')

## Step 5 - Neural RSMs (300x300)

**Why:** the neural RSM is the brain's word-geometry **fingerprint**: cell (i, j) = Pearson
similarity of the responses to word i and word j across all channels. Built per session and per
condition (word / gap) so we can ask whether semantic structure survives into the silent gap.

**Input:** `grand.words_s0/s1`, `grand.gaps_s0/s1` (300 x N).

**Output:** `neural` with `.words_s0/.words_s1/.gaps_s0/.gaps_s1` (each 300x300) + RSM heatmaps.

In [ ]:
neural = rsa.neural_rsms(grand)   # .words_s0/.words_s1/.gaps_s0/.gaps_s1 (300x300 each)

viz.plot_neural_rsm(neural.words_s0, neural.words_s1, title='Neural RSM (Words)', out=RESULTS / 'neural_rsm_words.png')
viz.plot_neural_rsm(neural.gaps_s0,  neural.gaps_s1,  title='Neural RSM (Gaps)',  out=RESULTS / 'neural_rsm_gaps.png')

## Step 6 - GPT-2 embeddings (13 layers, cached)

**Why:** GPT-2's representation of the same 300 words. Each word is encoded with a leading space
(`' word'`) for correct BPE context; the ~18 words that split into multiple tokens are mean-pooled
to one 768-d vector. All 13 layers (embedding + 12 transformer blocks) are kept. Cached to disk
so re-runs are instant.

**Input:** `grand.word_labels` (the 300 aligned words), `model`, `tokenizer`.

**Output:** `emb` - dict layer 0..12 -> (300 x 768); saved to `CACHE/gpt2_embeddings.npz`.

In [ ]:
emb = gpt2.embed_words(grand.word_labels, model, tokenizer, device,
                       cache=CACHE / 'gpt2_embeddings.npz')   # recompute only if cache missing
print(f'GPT-2 embeddings: 13 layers, shape per layer {emb[0].shape}')

## Step 7 - GPT-2 per-layer RSMs

**Why:** apply the **exact same metric** used on the neural side (pairwise Pearson over the 300
words) to each GPT-2 layer. Using an identical measure on both sides is what makes the comparison
fair, and it lets us track how word geometry evolves with model depth.

**Input:** the 13 embedding matrices.

**Output:** `gpt2_rsms` - dict layer 0..12 -> (300x300); sample-layer heatmaps.

In [ ]:
gpt2_rsms = gpt2.layer_rsms(emb)   # dict layer -> 300x300

viz.plot_gpt2_rsms(gpt2_rsms, layers=[0, 6, 12], out=RESULTS / 'gpt2_rsm_layers.png')   # embedding / middle / final

## Step 8 - Second-order RSA (neural vs each layer)

**Why:** the core comparison. Take the upper triangle of each RSM (44,850 unique word pairs) and
correlate the neural vector with each GPT-2 layer's vector, using **Pearson** (same similarity
scale?) and **Spearman** (same rank order?). Done per session and per condition (word/gap) to see
at which layer the brain-LM alignment peaks and whether it holds in the gap.

**Input:** `neural` RSMs + the 13 `gpt2_rsms`.

**Output:** per-layer Pearson & Spearman (S0 + S1) for words and gaps; layer-profile plots +
CSV tables.

In [ ]:
rsa_words = rsa.second_order_layers(neural.words_s0, neural.words_s1, gpt2_rsms)   # per-layer pearson + spearman, S0/S1
rsa_gaps  = rsa.second_order_layers(neural.gaps_s0,  neural.gaps_s1,  gpt2_rsms)

viz.plot_rsa_layer_profile(rsa_words, title='Neural-GPT-2 RSA (Words)', out=RESULTS / 'rsa_layer_profile_words.png')
viz.plot_rsa_layer_profile(rsa_gaps,  title='Neural-GPT-2 RSA (Gaps)',  out=RESULTS / 'rsa_layer_profile_gaps.png')

rsa.export_layer_table(rsa_words, out=RESULTS / 'rsa_results_words.csv')   # pearson + spearman, both sessions
rsa.export_layer_table(rsa_gaps,  out=RESULTS / 'rsa_results_gaps.csv')

## Step 9 - Permutation test (N=1000)

**Why:** the RSA values are small, so we must show they beat chance. We shuffle the word labels
of the neural RSM (rows + columns together) 1,000 times and recompute Spearman RSA against each
layer, building an empirical null. The empirical **p** = fraction of null >= observed; **z** =
(observed - null mean) / null std. Run per session; null distributions cached.

**Input:** `neural` RSMs + `gpt2_rsms`; `n_perms=1000`, `metric='spearman'`.

**Output:** per-layer p + z (S0/S1); null-band plot (observed vs null +/- 2 SD) and a best-layer
histogram, for words and gaps.

In [ ]:
perm_words = permutation.layer_permutation_test(neural.words_s0, neural.words_s1, gpt2_rsms,
                                                n_perms=1000, metric='spearman',
                                                cache=CACHE / 'perm_words.npz')
perm_gaps  = permutation.layer_permutation_test(neural.gaps_s0,  neural.gaps_s1,  gpt2_rsms,
                                                n_perms=1000, metric='spearman',
                                                cache=CACHE / 'perm_gaps.npz')

viz.plot_permutation_band(perm_words, out=RESULTS / 'permutation_band_words.png')
viz.plot_permutation_histogram(perm_words, out=RESULTS / 'permutation_hist_words.png')   # best layer
viz.plot_permutation_band(perm_gaps,  out=RESULTS / 'permutation_band_gaps.png')

## Outputs

Written under `results/gpt2/` (GPT-2 embeddings + null distributions cached in `results/gpt2/cache/`):

- **Neural RSMs (Step 5):** `neural_rsm_words.png`, `neural_rsm_gaps.png`
- **GPT-2 RSMs (Step 7):** `gpt2_rsm_layers.png`
- **RSA layer profile (Step 8):** `rsa_layer_profile_words.png`, `rsa_layer_profile_gaps.png`, `rsa_results_words.csv`, `rsa_results_gaps.csv`
- **Permutation test (Step 9):** `permutation_band_words.png`, `permutation_hist_words.png`, `permutation_band_gaps.png`

**To re-run without preprocessing:** point `IEEG_DATA_ROOT` at the processed-data tier; GPT-2
embeddings + null distributions reload from cache, so re-runs are fast.

*Deferred (say the word to add):* per-region breakdown (PFC / IFG / MTL / Temporal / Occipital /
Motor) x (word / gap) - the book's full 4-combination version.